## Imports and Setup

In [15]:
!pip install pdfplumber
!pip install python-docx

In [16]:
import re
import csv
import math
import pprint
import requests
import pandas as pd
import pdfplumber
from docx import Document
from docx.oxml.ns import qn
from collections import Counter

import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

True

## Task 1: Multi-Page PDF Extraction with Word Frequency Analysis

### (a) Extract text from each page of a PDF

In [18]:
pdf_path = "/content/Regex, lab 2,3 Tokenization & Sentence Segmentation.pdf"

pages_text = []
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text is None:
            text = ""
        pages_text.append(text)

print("Total number of pages extracted:", len(pages_text))

Total number of pages extracted: 4


### (b) Merge pages and tokenize

In [19]:
full_text = " ".join(pages_text)

cleaned_text = re.sub(r'[^a-zA-Z\s]', '', full_text)
cleaned_text = cleaned_text.lower()

tokens = cleaned_text.split()

print("Total token count:", len(tokens))

Total token count: 1243


### (c) Word frequency dictionary (top 10 words)

In [20]:
stopwords_list = [
    "the", "is", "in", "and", "to", "of", "a", "for", "on", "with",
    "as", "by", "at", "an", "be", "this", "that", "it", "from", "are"
]

filtered_tokens = [word for word in tokens if word not in stopwords_list]

word_freq = {}
for word in filtered_tokens:
    word_freq[word] = word_freq.get(word, 0) + 1

top_10_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 most frequent words:")
for word, count in top_10_words:
    print(f"{word}: {count}")

Top 10 most frequent words:
used: 24
nlp: 24
or: 22
pipeline: 20
text: 16
matters: 15
where: 15
why: 14
like: 14
can: 13


### (d) Search for a keyword and print matching sentences with page numbers

In [21]:
keyword = input("Enter a keyword to search: ")

found_match = False
for page_num, page_text in enumerate(pages_text, start=1):
    sentences = re.split(r'(?<=[.!?])\s+', page_text)
    for sentence in sentences:
        if keyword.lower() in sentence.lower():
            print(f"Page {page_num}: {sentence.strip()}")
            found_match = True

if not found_match:
    print(f"No sentence containing '{keyword}' was found.")

Enter a keyword to search: NLTK
Page 3: NLTK Tokenization
from nltk.tokenize import word_tokenize
word_tokenize("I don't like it.")
NLTK tokenization is simple, rule-based, and fast.
Page 3: NLTK vs spaCy - Comparison
Feature NLTK spaCy
Speed Medium Fast
Contractions (don't) Splits differently Handles smartly
Output type List of plain text words Token objects (with extra attributes)
Best used for Learning and research Real-world / production NLP
projects
Why it matters: The two tools can give different tokenization results.


## Task 2: Structured Page Metadata Extraction and CSV Export

### (a) Build page metadata dictionaries

In [24]:
pdf_path = "/content/Regex, lab 2,3 Tokenization & Sentence Segmentation.pdf"

pages_data = []
with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        raw_text = page.extract_text()
        if raw_text is None:
            raw_text = ""
        word_count = len(raw_text.split())
        char_count = len(raw_text.replace(" ", ""))
        pages_data.append({
            "page_number": i,
            "raw_text": raw_text,
            "word_count": word_count,
            "char_count": char_count
        })

print("First 2 entries of pages_data:")
for entry in pages_data[:2]:
    print(entry)

First 2 entries of pages_data:
{'page_number': 1, 'raw_text': 'Part A: Regex Fundamentals (Foundation)\nA1. What is Regex?\nRegex (Regular Expression) is a special pattern language. It is used to find specific pieces of text, such as emails,\nphone numbers, or dates, inside a big block of text.\nWhy it matters: Raw text (emails, chats, documents) contains useful information mixed with a lot of other text.\nWithout regex, we would have to search everything by hand, which is slow and full of mistakes.\nWhere it is used in NLP pipeline: Regex is used in the preprocessing step. Before any NLP model works on\nthe data, we clean the text or pull out specific information (like emails or phone numbers) using regex.\nA2. Character Classes\nCharacter classes tell regex what type of character to look for.\nSymbol Meaning\n\\d Any digit (0-9)\n\\w Any word character (letters, digits, underscore)\n\\s Space, tab, or new line\n[a-z] Any lowercase letter from a to z\n[A-Z0-9] A custom set - capital l

### (b) Highest / lowest word count pages and the average

In [25]:
highest_page = max(pages_data, key=lambda p: p["word_count"])
nonzero_pages = [p for p in pages_data if p["word_count"] > 0]
lowest_page = min(nonzero_pages, key=lambda p: p["word_count"]) if nonzero_pages else None

total_words = sum(p["word_count"] for p in pages_data)
average_word_count = round(total_words / len(pages_data), 2) if pages_data else 0

print("Page with highest word count:")
print(f"Page {highest_page['page_number']} - {highest_page['word_count']} words")
print(highest_page["raw_text"])

print("\nPage with lowest non-zero word count:")
if lowest_page:
    print(f"Page {lowest_page['page_number']} - {lowest_page['word_count']} words")
else:
    print("No page has a non-zero word count.")

print("\nAverage word count across all pages:", average_word_count)

Page with highest word count:
Page 1 - 381 words
Part A: Regex Fundamentals (Foundation)
A1. What is Regex?
Regex (Regular Expression) is a special pattern language. It is used to find specific pieces of text, such as emails,
phone numbers, or dates, inside a big block of text.
Why it matters: Raw text (emails, chats, documents) contains useful information mixed with a lot of other text.
Without regex, we would have to search everything by hand, which is slow and full of mistakes.
Where it is used in NLP pipeline: Regex is used in the preprocessing step. Before any NLP model works on
the data, we clean the text or pull out specific information (like emails or phone numbers) using regex.
A2. Character Classes
Character classes tell regex what type of character to look for.
Symbol Meaning
\d Any digit (0-9)
\w Any word character (letters, digits, underscore)
\s Space, tab, or new line
[a-z] Any lowercase letter from a to z
[A-Z0-9] A custom set - capital letters or digits
Why it matters:

### (c) Case-insensitive phrase search with a custom exception

In [26]:
class PhraseNotFoundError(Exception):
    pass

search_phrase = input("Enter a search phrase: ")

matches = []
for page in pages_data:
    count = page["raw_text"].lower().count(search_phrase.lower())
    if count > 0:
        matches.append((page["page_number"], count))

try:
    if not matches:
        raise PhraseNotFoundError(f"The phrase '{search_phrase}' was not found in any page.")
    for page_number, count in matches:
        print(f"Page {page_number}: '{search_phrase}' appears {count} time(s)")
except PhraseNotFoundError as e:
    print("Error:", e)

Enter a search phrase: spaCy's trained model (not the blank pipeline) handles these tricky cases better, because it has learned real language  patterns.
Error: The phrase 'spaCy's trained model (not the blank pipeline) handles these tricky cases better, because it has learned real language  patterns.' was not found in any page.


### (d) Write and re-read `pdf_summary.csv`

In [27]:
with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["page_number", "word_count", "char_count"])
    for page in pages_data:
        writer.writerow([page["page_number"], page["word_count"], page["char_count"]])

print("pdf_summary.csv written successfully.\n")

print("Contents of pdf_summary.csv:")
with open("pdf_summary.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

pdf_summary.csv written successfully.

Contents of pdf_summary.csv:
['page_number', 'word_count', 'char_count']
['1', '381', '1792']
['2', '266', '1540']
['3', '329', '1825']
['4', '300', '1758']


## Task 3: PDF Text Preprocessing Pipeline for NLP

### (a) Extract text and clean it (remove URLs, digits, non-alphabetic characters)

In [28]:
pdf_path = "/content/Regex, lab 2,3 Tokenization & Sentence Segmentation.pdf"  # update this path

full_text = ""
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            full_text += text + " "

char_count_before = len(full_text)

cleaned = re.sub(r'http\S+|www\.\S+', '', full_text)          # remove URLs
cleaned = re.sub(r'\d+', '', cleaned)                           # remove digits
cleaned = re.sub(r'[^a-zA-Z\s.]', '', cleaned)                  # keep letters, spaces, full stops
cleaned = re.sub(r'\s+', ' ', cleaned).strip()                  # collapse multiple spaces

char_count_after = len(cleaned)

print("Character count before cleaning:", char_count_before)
print("Character count after cleaning:", char_count_after)

Character count before cleaning: 8053
Character count after cleaning: 7569


### (b) Lowercase, split into sentences, remove empty strings

In [29]:
cleaned_lower = cleaned.lower()

sentences = re.split(r'[.!?]', cleaned_lower)
sentences = [s.strip() for s in sentences]

sentence_count_before = len(sentences)

sentences = [s for s in sentences if s != ""]

sentence_count_after = len(sentences)

print("Sentence count before removing empty strings:", sentence_count_before)
print("Sentence count after removing empty strings:", sentence_count_after)

Sentence count before removing empty strings: 110
Sentence count after removing empty strings: 109


### (c) Tokenize, remove stopwords, and filter short sentences

In [30]:
english_stopwords = set(stopwords.words('english'))

sentence_count_before_filter = len(sentences)

filtered_sentences = []
for sentence in sentences:
    words = sentence.split()
    words_no_stopwords = [w for w in words if w not in english_stopwords]
    if len(words_no_stopwords) >= 5:
        filtered_sentences.append(sentence)

sentence_count_after_filter = len(filtered_sentences)

print("Sentence count before stopword filter:", sentence_count_before_filter)
print("Sentence count after stopword filter:", sentence_count_after_filter)

print("\n3 sample sentences:")
for s in filtered_sentences[:3]:
    print("-", s)

Sentence count before stopword filter: 109
Sentence count after stopword filter: 64

3 sample sentences:
- what is regex regex regular expression is a special pattern language
- it is used to find specific pieces of text such as emails phone numbers or dates inside a big block of text
- why it matters raw text emails chats documents contains useful information mixed with a lot of other text


### (d) Write to `nlp_ready_output.txt` and analyze it

In [31]:
with open("nlp_ready_output.txt", "w", encoding="utf-8") as f:
    for sentence in filtered_sentences:
        f.write(sentence + "\n")

with open("nlp_ready_output.txt", "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f.readlines()]

longest_sentence = max(lines, key=len) if lines else ""
shortest_sentence = min(lines, key=len) if lines else ""

print("Total lines written:", len(lines))
print("\nLongest sentence:", longest_sentence)
print("\nShortest sentence:", shortest_sentence)

Total lines written: 64

Longest sentence: quick recap table pipeline order step topic pipeline stage regex fundamentals preprocessing cleaning patternbased extraction information extraction word tokenization text into tokens token attributes feature engineering filtering sentence segmentation tokens into sentencelevel structure this order matters because in a real nlp system text is first cleaned and important information is extracted then it is tokenized then we check the properties of each token and finally the text is organized into sentences

Shortest sentence: part c word tokenization c


## Task 4: DOCX Paragraph Classification and Statistical Analysis

### (a) Extract paragraph text and style names

In [33]:
docx_path = "/content/Lab 02 Task.docx"  # update this path

doc = Document(docx_path)

paragraphs_info = []
for para in doc.paragraphs:
    paragraphs_info.append({
        "text": para.text,
        "style": para.style.name
    })

unique_styles = set(p["style"] for p in paragraphs_info)

print("Total paragraph count:", len(paragraphs_info))
print("Unique style names found:", unique_styles)

Total paragraph count: 46
Unique style names found: {'Normal'}


### (b) Classify paragraphs into Headings, Body, and Other/Empty

In [34]:
headings = []
body = []
other = []

for p in paragraphs_info:
    style = p["style"]
    if style.startswith("Heading"):
        headings.append(p)
    elif style == "Normal" or "Body" in style:
        body.append(p)
    else:
        other.append(p)

print("Headings count:", len(headings))
print("Body count:", len(body))
print("Other/Empty count:", len(other))

Headings count: 0
Body count: 46
Other/Empty count: 0


### (c) Statistics for Body paragraphs with more than 5 words

In [35]:
qualifying_body = []
for p in body:
    word_count = len(p["text"].split())
    if word_count > 5:
        char_count = len(p["text"].replace(" ", ""))
        qualifying_body.append({
            "text": p["text"],
            "word_count": word_count,
            "char_count": char_count
        })

if qualifying_body:
    highest = max(qualifying_body, key=lambda x: x["word_count"])
    lowest = min(qualifying_body, key=lambda x: x["word_count"])
    mean_word_count = round(sum(p["word_count"] for p in qualifying_body) / len(qualifying_body), 2)

    print("Paragraph with highest word count:")
    print(highest["text"], "->", highest["word_count"], "words")

    print("\nParagraph with lowest word count (above 5):")
    print(lowest["text"], "->", lowest["word_count"], "words")

    print("\nMean word count:", mean_word_count)
else:
    print("No body paragraphs with more than 5 words were found.")

Paragraph with highest word count:
(a) Fetch data from the endpoint below and parse it into a Python list. Validate that: (i) the response status is 200, (ii) the parsed result is a list, and (iii) the list contains at least one record. If any validation fails, print a meaningful error and exit. Print the total number of todo records retrieved.
https://jsonplaceholder.typicode.com/todos -> 57 words

Paragraph with lowest word count (above 5):
Task 3: PDF Text Preprocessing Pipeline for NLP -> 8 words

Mean word count: 35.93


### (d) Nested dictionary of styles, sorted by count

In [36]:
style_dict = {}
for p in paragraphs_info:
    style = p["style"]
    if style not in style_dict:
        style_dict[style] = {"count": 0, "texts": []}
    style_dict[style]["count"] += 1
    style_dict[style]["texts"].append(p["text"])

for style in style_dict:
    if style_dict[style]["count"] > 2:
        style_dict[style]["texts"] = style_dict[style]["texts"][:2]

sorted_style_dict = dict(
    sorted(style_dict.items(), key=lambda x: x[1]["count"], reverse=True)
)

pprint.pprint(sorted_style_dict)

{'Normal': {'count': 46,
            'texts': ['Task 1: Multi-Page PDF Extraction with Word Frequency '
                      'Analysis',
                      '(a) Open a PDF file using PyPDF2 or pdfplumber and '
                      'extract text from each page individually. Store the '
                      'result as a list of strings where each index represents '
                      'one page. Print the total number of pages extracted.']}}


## Task 5: DOCX Table Extraction and Pandas Integration

### (a) Extract all tables from the document

In [43]:
docx_path = "/content/Lab05.docx"  # update this path (must contain at least one table)

doc = Document(docx_path)

all_tables = []
for table in doc.tables:
    table_data = []
    for row in table.rows:
        row_data = [cell.text for cell in row.cells]
        table_data.append(row_data)
    all_tables.append(table_data)

print("Total number of tables found:", len(all_tables))
for i, table_data in enumerate(all_tables, start=1):
    rows = len(table_data)
    cols = len(table_data[0]) if rows > 0 else 0
    print(f"Table {i}: {rows} rows x {cols} columns")

Total number of tables found: 9
Table 1: 5 rows x 3 columns
Table 2: 3 rows x 4 columns
Table 3: 5 rows x 3 columns
Table 4: 5 rows x 5 columns
Table 5: 5 rows x 2 columns
Table 6: 4 rows x 2 columns
Table 7: 3 rows x 2 columns
Table 8: 7 rows x 3 columns
Table 9: 7 rows x 3 columns


### (b) Convert the first table into a pandas DataFrame

In [44]:
first_table = all_tables[0]
header = first_table[0]
data_rows = first_table[1:]

df = pd.DataFrame(data_rows, columns=header)
print(df)

print("\nEmpty or None cells per column:")
empty_counts = df.apply(lambda col: col.isnull().sum() + (col == "").sum())
print(empty_counts)

           Feature                                         ADALINE  \
0   Training error                   Uses continuous linear output   
1  Learning method  Delta rule / gradient descent on squared error   
2         Decision            Threshold applied for classification   
3         Strength                          Smooth error objective   

                            Perceptron  
0          Uses thresholded prediction  
1               Perceptron update rule  
2  Step function is part of prediction  
3      Simple online linear classifier  

Empty or None cells per column:
Feature       0
ADALINE       0
Perceptron    0
dtype: int64


### (c) Extract paragraphs that appear before the first table

In [45]:
pre_table_paragraphs = []
for child in doc.element.body.iterchildren():
    if child.tag == qn('w:tbl'):
        break
    if child.tag == qn('w:p'):
        text = "".join(node.text for node in child.iter(qn('w:t')) if node.text)
        pre_table_paragraphs.append(text)

print("Number of pre-table paragraphs:", len(pre_table_paragraphs))

Number of pre-table paragraphs: 36


### (d) Write pre-table paragraphs and table data to `docx_extracted.txt`

In [46]:
with open("docx_extracted.txt", "w", encoding="utf-8") as f:
    f.write("--- PRE-TABLE PARAGRAPHS ---\n")
    for para in pre_table_paragraphs:
        f.write(para + "\n")

    f.write("\n--- TABLE DATA ---\n")
    for row in first_table:
        f.write(" | ".join(row) + "\n")

print("docx_extracted.txt written successfully.")

docx_extracted.txt written successfully.


## Task 6: Multi-Document Comparison, Deduplication, and Frequency Report

### (a) Compare headings between two documents

In [47]:
docx_path_a = "/content/Lab05.docx"  # update this path
docx_path_b = "/content/Lab 02 Task.docx"  # update this path

doc_a = Document(docx_path_a)
doc_b = Document(docx_path_b)

headings_a = [p.text for p in doc_a.paragraphs if p.style.name.startswith("Heading")]
headings_b = [p.text for p in doc_b.paragraphs if p.style.name.startswith("Heading")]

set_a = set(headings_a)
set_b = set(headings_b)

print("Headings common to both documents:", set_a & set_b)
print("Headings only in Document A:", set_a - set_b)
print("Headings only in Document B:", set_b - set_a)

Headings common to both documents: set()
Headings only in Document A: {'7)  Graded Lab Tasks', 'Graded Task 2 — MADALINE', 'Activity Outcomes', 'Required software: Python, NumPy, Matplotlib, and scikit-learn.', 'Graded Task 1 — ADALINE', 'Submission Checklist', '5)  Useful Concepts', 'Part A — ADALINE Neural Network', 'Part B — MADALINE Neural Network', 'Objective', 'Activity 2: MADALINE-Style Classifier on Iris Dataset', '6)  Solved Lab Activities', 'Activity 1: Train ADALINE for the AND Gate'}
Headings only in Document B: set()


### (b) Merge Normal paragraphs and remove duplicates

In [48]:
normal_a = [p.text for p in doc_a.paragraphs if p.style.name == "Normal" and p.text.strip() != ""]
normal_b = [p.text for p in doc_b.paragraphs if p.style.name == "Normal" and p.text.strip() != ""]

merged_paragraphs = normal_a + normal_b
count_before = len(merged_paragraphs)

seen = set()
deduplicated_paragraphs = []
for para in merged_paragraphs:
    key = para.strip().lower()
    if key not in seen:
        seen.add(key)
        deduplicated_paragraphs.append(para)

count_after = len(deduplicated_paragraphs)

print("Total paragraphs before deduplication:", count_before)
print("Total paragraphs after deduplication:", count_after)

Total paragraphs before deduplication: 238
Total paragraphs after deduplication: 226


### (c) Combined word frequency (top 20 words)

In [49]:
stopwords_15 = [
    "the", "is", "in", "and", "to", "of", "a", "for", "on", "with",
    "as", "by", "at", "an", "be"
]

all_words = []
for para in deduplicated_paragraphs:
    cleaned_para = re.sub(r'[^a-zA-Z\s]', '', para).lower()
    words = cleaned_para.split()
    all_words.extend(words)

filtered_words = [w for w in all_words if w not in stopwords_15]

word_freq = Counter(filtered_words)
top_20_words = word_freq.most_common(20)

print("Top 20 most frequent words:")
for word, count in top_20_words:
    print(f"{word}: {count}")

Top 20 most frequent words:
print: 36
count: 25
each: 24
b: 22
all: 22
output: 20
hidden: 20
adaline: 19
from: 19
using: 18
list: 18
learning: 15
error: 14
x: 14
into: 14
ii: 14
total: 14
linear: 13
task: 13
training: 13


### (d) Summary report

In [50]:
total_headings = len(headings_a) + len(headings_b)
total_unique_body_paragraphs = len(deduplicated_paragraphs)
vocabulary_size = len(set(filtered_words))

paragraph_lengths = [len(p.split()) for p in deduplicated_paragraphs]
average_paragraph_length = round(sum(paragraph_lengths) / len(paragraph_lengths), 2) if paragraph_lengths else 0

longest_paragraph = max(deduplicated_paragraphs, key=lambda p: len(p.split())) if deduplicated_paragraphs else ""

print("Total headings across both documents:", total_headings)
print("Total unique body paragraphs after deduplication:", total_unique_body_paragraphs)
print("Total unique word types (vocabulary size):", vocabulary_size)
print("Average paragraph length in words:", average_paragraph_length)
print("Longest paragraph by word count:")
print(longest_paragraph)

Total headings across both documents: 13
Total unique body paragraphs after deduplication: 226
Total unique word types (vocabulary size): 851
Average paragraph length in words: 13.74
Longest paragraph by word count:
(a) Fetch data from the endpoint below and parse it into a Python list. Validate that: (i) the response status is 200, (ii) the parsed result is a list, and (iii) the list contains at least one record. If any validation fails, print a meaningful error and exit. Print the total number of todo records retrieved.
https://jsonplaceholder.typicode.com/todos


## Task 7: Multi-Endpoint API Data Joining and Analysis

### (a) Fetch users and posts, validating the status code

In [52]:
users_url = "https://jsonplaceholder.typicode.com/users"
posts_url = "https://jsonplaceholder.typicode.com/posts"

users_response = requests.get(users_url)
if users_response.status_code != 200:
    raise ConnectionError(f"Failed to fetch users. Status code: {users_response.status_code}")
users_data = users_response.json()

posts_response = requests.get(posts_url)
if posts_response.status_code != 200:
    raise ConnectionError(f"Failed to fetch posts. Status code: {posts_response.status_code}")
posts_data = posts_response.json()

print("Users fetched:", len(users_data))
print("Posts fetched:", len(posts_data))

Users fetched: 10
Posts fetched: 100


### (b) Join users with their posts

In [53]:
joined_data = {}
for user in users_data:
    user_posts = [post["title"] for post in posts_data if post["userId"] == user["id"]]
    joined_data[user["name"]] = {
        "email": user["email"],
        "post_count": len(user_posts),
        "titles": user_posts
    }

print("Joined data for the first 3 users:")
for name in list(joined_data.keys())[:3]:
    print(name, "->", joined_data[name])

Joined data for the first 3 users:
Leanne Graham -> {'email': 'Sincere@april.biz', 'post_count': 10, 'titles': ['sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'qui est esse', 'ea molestias quasi exercitationem repellat qui ipsa sit aut', 'eum et est occaecati', 'nesciunt quas odio', 'dolorem eum magni eos aperiam quia', 'magnam facilis autem', 'dolorem dolore est ipsam', 'nesciunt iure omnis dolorem tempora et accusantium', 'optio molestias id quia eum']}
Ervin Howell -> {'email': 'Shanna@melissa.tv', 'post_count': 10, 'titles': ['et ea vero quia laudantium autem', 'in quibusdam tempore odit est dolorem', 'dolorum ut in voluptas mollitia et saepe quo animi', 'voluptatem eligendi optio', 'eveniet quod temporibus', 'sint suscipit perspiciatis velit dolorum rerum ipsa laboriosam odio', 'fugit voluptas sed molestias voluptatem provident', 'voluptate et itaque vero tempora molestiae', 'adipisci placeat illum aut reiciendis qui', 'doloribus ad provident suscipi

### (c) User with the most posts

In [54]:
top_user_name = max(joined_data, key=lambda name: joined_data[name]["post_count"])
top_user_info = joined_data[top_user_name]

top_user_full = next(u for u in users_data if u["name"] == top_user_name)

print("Full name:", top_user_name)
print("Email:", top_user_info["email"])
print("Company name:", top_user_full["company"]["name"])
print("\nAll post titles:")
for i, title in enumerate(top_user_info["titles"], start=1):
    print(f"{i}. {title}")

Full name: Leanne Graham
Email: Sincere@april.biz
Company name: Romaguera-Crona

All post titles:
1. sunt aut facere repellat provident occaecati excepturi optio reprehenderit
2. qui est esse
3. ea molestias quasi exercitationem repellat qui ipsa sit aut
4. eum et est occaecati
5. nesciunt quas odio
6. dolorem eum magni eos aperiam quia
7. magnam facilis autem
8. dolorem dolore est ipsam
9. nesciunt iure omnis dolorem tempora et accusantium
10. optio molestias id quia eum


### (d) Filter users by company buzzword or long city name

In [55]:
matching_users = []
for user in users_data:
    bs_text = user["company"]["bs"].lower()
    city = user["address"]["city"]
    if "synergies" in bs_text or len(city) > 8:
        post_count = joined_data[user["name"]]["post_count"]
        matching_users.append((user["name"], city, post_count))

print("Users matching the filter conditions:")
for name, city, post_count in matching_users:
    print(f"Name: {name}, City: {city}, Post Count: {post_count}")

Users matching the filter conditions:
Name: Leanne Graham, City: Gwenborough, Post Count: 10
Name: Ervin Howell, City: Wisokyburgh, Post Count: 10
Name: Clementine Bauch, City: McKenziehaven, Post Count: 10
Name: Patricia Lebsack, City: South Elvis, Post Count: 10
Name: Chelsey Dietrich, City: Roscoeview, Post Count: 10
Name: Mrs. Dennis Schulist, City: South Christy, Post Count: 10
Name: Kurtis Weissnat, City: Howemouth, Post Count: 10
Name: Nicholas Runolfsdottir V, City: Aliyaview, Post Count: 10
Name: Glenna Reichert, City: Bartholomebury, Post Count: 10
Name: Clementina DuBuque, City: Lebsackbury, Post Count: 10


## Task 8: To-Do Completion Rate Analysis Across Users

### (a) Fetch and validate the todos data

In [56]:
todos_url = "https://jsonplaceholder.typicode.com/todos"

response = requests.get(todos_url)

if response.status_code != 200:
    print("Error: could not fetch todos. Status code:", response.status_code)
else:
    todos_data = response.json()
    if not isinstance(todos_data, list):
        print("Error: response is not a list.")
    elif len(todos_data) == 0:
        print("Error: no todo records found.")
    else:
        print("Total number of todo records retrieved:", len(todos_data))

Total number of todo records retrieved: 200


### (b) Group todos by user and verify totals

In [57]:
user_todo_summary = {}

for todo in todos_data:
    uid = todo["userId"]
    if uid not in user_todo_summary:
        user_todo_summary[uid] = {"total": 0, "completed": 0, "incomplete": 0}
    user_todo_summary[uid]["total"] += 1
    if todo["completed"]:
        user_todo_summary[uid]["completed"] += 1
    else:
        user_todo_summary[uid]["incomplete"] += 1

for uid in sorted(user_todo_summary.keys()):
    stats = user_todo_summary[uid]
    check = (stats["completed"] + stats["incomplete"]) == stats["total"]
    print(f"User {uid}: {stats} -> Valid: {check}")

User 1: {'total': 20, 'completed': 11, 'incomplete': 9} -> Valid: True
User 2: {'total': 20, 'completed': 8, 'incomplete': 12} -> Valid: True
User 3: {'total': 20, 'completed': 7, 'incomplete': 13} -> Valid: True
User 4: {'total': 20, 'completed': 6, 'incomplete': 14} -> Valid: True
User 5: {'total': 20, 'completed': 12, 'incomplete': 8} -> Valid: True
User 6: {'total': 20, 'completed': 6, 'incomplete': 14} -> Valid: True
User 7: {'total': 20, 'completed': 9, 'incomplete': 11} -> Valid: True
User 8: {'total': 20, 'completed': 11, 'incomplete': 9} -> Valid: True
User 9: {'total': 20, 'completed': 8, 'incomplete': 12} -> Valid: True
User 10: {'total': 20, 'completed': 12, 'incomplete': 8} -> Valid: True


### (c) Completion rate per user, highest/lowest, and overall average

In [58]:
completion_rates = {}
for uid, stats in user_todo_summary.items():
    rate = round((stats["completed"] / stats["total"]) * 100, 1)
    completion_rates[uid] = rate

highest_uid = max(completion_rates, key=completion_rates.get)
lowest_uid = min(completion_rates, key=completion_rates.get)
overall_average = round(sum(completion_rates.values()) / len(completion_rates), 1)

print("Highest completion rate:", f"User {highest_uid} -> {completion_rates[highest_uid]}%")
print("Lowest completion rate:", f"User {lowest_uid} -> {completion_rates[lowest_uid]}%")
print("Overall average completion rate:", f"{overall_average}%")

Highest completion rate: User 5 -> 60.0%
Lowest completion rate: User 4 -> 30.0%
Overall average completion rate: 45.0%


### (d) Ranked table of users by completion rate

In [59]:
users_response = requests.get("https://jsonplaceholder.typicode.com/users")
users_lookup_data = users_response.json()

user_lookup = {user["id"]: user["name"] for user in users_lookup_data}

table_rows = []
for uid, stats in user_todo_summary.items():
    table_rows.append({
        "userId": uid,
        "name": user_lookup.get(uid, "Unknown"),
        "total": stats["total"],
        "completed": stats["completed"],
        "rate": completion_rates[uid]
    })

table_rows.sort(key=lambda x: x["rate"], reverse=True)

print(f"{'Rank':<6}{'Name':<20}{'Total Tasks':<14}{'Completed':<12}{'Completion Rate %':<18}")
for rank, row in enumerate(table_rows, start=1):
    print(f"{rank:<6}{row['name']:<20}{row['total']:<14}{row['completed']:<12}{row['rate']:<18}")

Rank  Name                Total Tasks   Completed   Completion Rate % 
1     Chelsey Dietrich    20            12          60.0              
2     Clementina DuBuque  20            12          60.0              
3     Leanne Graham       20            11          55.0              
4     Nicholas Runolfsdottir V20            11          55.0              
5     Kurtis Weissnat     20            9           45.0              
6     Ervin Howell        20            8           40.0              
7     Glenna Reichert     20            8           40.0              
8     Clementine Bauch    20            7           35.0              
9     Patricia Lebsack    20            6           30.0              
10    Mrs. Dennis Schulist20            6           30.0              


## Task 9: Nested JSON Flattening, DataFrame Operations, and Comment Analysis

### (a) Flatten nested user objects

In [60]:
users_response = requests.get("https://jsonplaceholder.typicode.com/users")
users_data = users_response.json()

def flatten_user(user):
    return {
        "id": user["id"],
        "name": user["name"],
        "username": user["username"],
        "email": user["email"],
        "city": user["address"]["city"],
        "zipcode": user["address"]["zipcode"],
        "lat": user["address"]["geo"]["lat"],
        "lng": user["address"]["geo"]["lng"],
        "company_name": user["company"]["name"],
        "company_catchphrase": user["company"]["catchPhrase"]
    }

flattened_users = [flatten_user(user) for user in users_data]

print("Flattened user list:")
for user in flattened_users:
    print(user)

Flattened user list:
{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'lat': '-37.3159', 'lng': '81.1496', 'company_name': 'Romaguera-Crona', 'company_catchphrase': 'Multi-layered client-server neural-net'}
{'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'lat': '-43.9509', 'lng': '-34.4618', 'company_name': 'Deckow-Crist', 'company_catchphrase': 'Proactive didactic contingency'}
{'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'city': 'McKenziehaven', 'zipcode': '59590-4157', 'lat': '-68.6102', 'lng': '-47.0653', 'company_name': 'Romaguera-Jacobson', 'company_catchphrase': 'Face to face bifurcated interface'}
{'id': 4, 'name': 'Patricia Lebsack', 'username': 'Karianne', 'email': 'Julianne.OConner@kory.org', 'city': 'South Elvis', 'zipcode': '53919-4257', 'lat': '29.4572', 

### (b) DataFrame operations and distance from origin

In [61]:
df_users = pd.DataFrame(flattened_users)
print(df_users)

df_users["lat"] = df_users["lat"].astype(float)
df_users["lng"] = df_users["lng"].astype(float)

df_users["distance_from_origin"] = df_users.apply(
    lambda row: math.sqrt(row["lat"] ** 2 + row["lng"] ** 2), axis=1
)

farthest_user = df_users.loc[df_users["distance_from_origin"].idxmax()]
print("\nUser with the greatest distance from origin:")
print(farthest_user)

   id                      name          username                      email  \
0   1             Leanne Graham              Bret          Sincere@april.biz   
1   2              Ervin Howell         Antonette          Shanna@melissa.tv   
2   3          Clementine Bauch          Samantha         Nathan@yesenia.net   
3   4          Patricia Lebsack          Karianne  Julianne.OConner@kory.org   
4   5          Chelsey Dietrich            Kamren   Lucio_Hettinger@annie.ca   
5   6      Mrs. Dennis Schulist  Leopoldo_Corkery    Karley_Dach@jasper.info   
6   7           Kurtis Weissnat      Elwyn.Skiles     Telly.Hoeger@billy.biz   
7   8  Nicholas Runolfsdottir V     Maxime_Nienow       Sherwood@rosamond.me   
8   9           Glenna Reichert          Delphine    Chaim_McDermott@dana.io   
9  10        Clementina DuBuque    Moriah.Stanton     Rey.Padberg@karina.biz   

             city     zipcode       lat        lng        company_name  \
0     Gwenborough  92998-3874  -37.3159    81

### (c) Northern vs Southern Hemisphere comparison

In [62]:
northern_users = df_users[df_users["lat"] > 0]["name"].tolist()
southern_users = df_users[df_users["lat"] < 0]["name"].tolist()

print("Northern Hemisphere users:", northern_users)
print("Southern Hemisphere users:", southern_users)

northern_avg_distance = df_users[df_users["lat"] > 0]["distance_from_origin"].mean()
southern_avg_distance = df_users[df_users["lat"] < 0]["distance_from_origin"].mean()

if northern_avg_distance > southern_avg_distance:
    print("\nNorthern Hemisphere has the higher average distance from origin.")
else:
    print("\nSouthern Hemisphere has the higher average distance from origin.")

Northern Hemisphere users: ['Patricia Lebsack', 'Kurtis Weissnat', 'Glenna Reichert']
Southern Hemisphere users: ['Leanne Graham', 'Ervin Howell', 'Clementine Bauch', 'Chelsey Dietrich', 'Mrs. Dennis Schulist', 'Nicholas Runolfsdottir V', 'Clementina DuBuque']

Northern Hemisphere has the higher average distance from origin.


### (d) Comment counts per post — top 5 and bottom 5

In [63]:
posts_response = requests.get("https://jsonplaceholder.typicode.com/posts")
posts_data = posts_response.json()

comments_response = requests.get("https://jsonplaceholder.typicode.com/comments")
comments_data = comments_response.json()

comment_counts = {}
for comment in comments_data:
    pid = comment["postId"]
    comment_counts[pid] = comment_counts.get(pid, 0) + 1

post_comment_list = []
for post in posts_data:
    pid = post["id"]
    count = comment_counts.get(pid, 0)
    post_comment_list.append((pid, post["title"][:40], count))

post_comment_list.sort(key=lambda x: x[2], reverse=True)

print("Top 5 most-commented posts:")
for pid, title, count in post_comment_list[:5]:
    print(f"Post {pid}: '{title}' -> {count} comments")

print("\nBottom 5 least-commented posts:")
for pid, title, count in post_comment_list[-5:]:
    print(f"Post {pid}: '{title}' -> {count} comments")

Top 5 most-commented posts:
Post 1: 'sunt aut facere repellat provident occae' -> 5 comments
Post 2: 'qui est esse' -> 5 comments
Post 3: 'ea molestias quasi exercitationem repell' -> 5 comments
Post 4: 'eum et est occaecati' -> 5 comments
Post 5: 'nesciunt quas odio' -> 5 comments

Bottom 5 least-commented posts:
Post 96: 'quaerat velit veniam amet cupiditate aut' -> 5 comments
Post 97: 'quas fugiat ut perspiciatis vero provide' -> 5 comments
Post 98: 'laboriosam dolor voluptates' -> 5 comments
Post 99: 'temporibus sit alias delectus eligendi p' -> 5 comments
Post 100: 'at nam consequatur ea labore ea harum' -> 5 comments
